<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/deeepseeklocalepynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guida Semplice per l'installazione di DeepSeek Locale
Questa guida ti aiuterà a compilare e avviare il server anche se sei un principiante.

In [21]:
# 1. Compiliamo solo il server con impostazioni minimali per risparmiare RAM
!apt-get update && apt-get install cmake -y
%cd /content/llama.cpp
!mkdir -p build
%cd build
# Disabilitiamo TOOLS, TEST, EXAMPLES e feature pesanti per evitare crash
!cmake .. -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=ON -DGGML_CCACHE=OFF -DGGML_OPENMP=OFF -DLLAMA_SERVER_SSL=OFF
# Compiliamo solo il target llama-server usando 1 solo thread
!cmake --build . --config Release --target llama-server -j 1

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,398 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubunt

In [53]:
# 2. Scarichiamo il modello DeepSeek-V3 Q2
import os
from huggingface_hub import hf_hub_download

model_dir = '/content/llama.cpp/models/deepseek'
os.makedirs(model_dir, exist_ok=True)

# Repository e nome file corretti
repo_id = "unsloth/DeepSeek-V3-GGUF"
model_filename = "DeepSeek-V3-Q2_K.gguf"
full_path = os.path.join(model_dir, model_filename)

print(f"Inizio download del modello (25GB) da {repo_id}...")

if os.path.exists(full_path):
    os.remove(full_path)

try:
    # Tentativo con huggingface_hub
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=model_filename,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )
    print("Download completato con successo!")
except Exception as e:
    print(f"Errore durante il download standard: {e}")
    print("Provo il download accelerato con aria2...")
    !apt-get install -y aria2
    url = f"https://huggingface.co/{repo_id}/resolve/main/{model_filename}"
    !aria2c -x 16 -s 16 -c -d {model_dir} -o {model_filename} "{url}"

print("Verifica presenza file:")
!ls -lh {full_path}

Inizio download del modello (25GB) da unsloth/DeepSeek-V3-GGUF...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Errore durante il download standard: 404 Client Error. (Request ID: Root=1-6a00ff4a-05266913388630080c3de83b;ed0de0e8-4c80-4809-a912-2f1a03969713)

Entry Not Found for url: https://huggingface.co/unsloth/DeepSeek-V3-GGUF/resolve/main/DeepSeek-V3-Q2_K.gguf.
Provo il download accelerato con aria2...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 17 not upgraded.

05/10 21:57:34 [NOTICE] Downloading 1 item(s)

05/10 21:57:34 [ERROR] CUID#7 - Download aborted. URI=https://huggingface.co/unsloth/DeepSeek-V3-GGUF/resolve/main/DeepSeek-V3-Q2_K.gguf
Exception: [AbstractCommand.cc:351] errorCode=3 URI=https://huggingface.co/unsloth/DeepSeek-V3-GGUF/resolve/main/DeepSeek-V3-Q2_K.gguf
  -> [HttpSkipResponseCommand.cc:218] errorCode=3 Resource not found

05/10 21:57:34 [NOTICE] Download GID#424c168368ca28e1 not complete: /content/llama.cpp/models/deepseek

In [23]:
# 3. Avviamo il server sulla porta 8000
import subprocess
import time
import os

# Verifichiamo i percorsi comuni dove viene creato il binario
paths = [
    "/content/llama.cpp/build/bin/llama-server",
    "/content/llama.cpp/build/llama-server"
]

server_bin = next((p for p in paths if os.path.exists(p)), None)
model_path = "/content/llama.cpp/models/deepseek/DeepSeek-V3-Q2_K.gguf"

if server_bin and os.path.exists(model_path):
    # Parametri conservativi per 12GB di RAM
    command = f"{server_bin} -m {model_path} --port 8000 --ctx-size 2048 --host 127.0.0.1 -n 128"
    with open("/content/server.log", "w") as log_file:
        subprocess.Popen(command.split(), stdout=log_file, stderr=log_file)
    print(f"Il server si sta avviando da: {server_bin}")
    print("Attendi 25 secondi...")
    time.sleep(25)
    !curl -s http://127.0.0.1:8000/v1/models || echo 'Il server non è ancora pronto. Controlla /content/server.log per dettagli.'
else:
    print('Errore: Compilazione non riuscita o modello non trovato. Riesegui i passaggi 1 e 2.')

Errore: Compilazione non riuscita o modello non trovato. Riesegui i passaggi 1 e 2.


# Step 1: Clone the Repository
To use scripts like `download_model.sh` and build the project, we need to clone the source code first.

In [6]:
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 92940, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 92940 (delta 268), reused 196 (delta 196), pack-reused 92594 (from 3)
Receiving objects: 100% (92940/92940), 386.65 MiB | 22.03 MiB/s, done.
Resolving deltas: 100% (65956/65956), done.
Updating files: 100% (2734/2734), done.
/content/llama.cpp


# Model Download Setup

Based on your resources, the model download script depends on the available RAM:
- `q2`: 128 GB RAM
- `q4`: >= 256 GB RAM

Let's check the available RAM on this instance first.

In [1]:
!free -h --si

               total        used        free      shared  buff/cache   available
Mem:             12G        587M        9.3G        1.0M        3.1G         12G
Swap:             0B          0B          0B


Now, you can run the appropriate command. By default, I will provide the `q2` command below, but you can change it to `q4` if your RAM allows.

In [4]:
!chmod +x ./models/download-ggml-model.sh
# Given your RAM is ~12GB, we should check if a small model is available.
# !./models/download-ggml-model.sh [MODEL_NAME]

chmod: cannot access './models/download-ggml-model.sh': No such file or directory


In [5]:
# Building the project so that binaries like 'main' or 'ds4' are created
!make -j

make: *** No targets specified and no makefile found.  Stop.


# Step 4: Run the Model
Now that the model is downloaded and the project is built, we can run the specific prompt.

In [7]:
# Running the specific command requested
!./ds4 -p "Explain Redis streams in one paragraph."

/bin/bash: line 1: ./ds4: No such file or directory


In [8]:
!git clone https://github.com/ggerganov/llama.cpp.git llama_repo
%cd llama_repo
!make -j
!./ds4-server --ctx 100000 --kv-disk-dir /tmp/ds4-kv --kv-disk-space-mb 8192

Cloning into 'llama_repo'...
remote: Enumerating objects: 92940, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 92940 (delta 268), reused 196 (delta 196), pack-reused 92594 (from 3)
Receiving objects: 100% (92940/92940), 386.65 MiB | 23.83 MiB/s, done.
Resolving deltas: 100% (65956/65956), done.
Updating files: 100% (2734/2734), done.
/content/llama.cpp/llama_repo
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
/bin/bash: line 1: ./ds4-server: No such file or directory


In [10]:
!curl http://127.0.0.1:8080/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{
    "model":"deepseek-v4-flash",
    "messages":[{"role":"user","content":"List three Redis design principles."}],
    "stream":true
  }'

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 5)

In [11]:
!./ds4-server --ctx 100000 --kv-disk-dir /tmp/ds4-kv --kv-disk-space-mb 8192

/bin/bash: line 1: ./ds4-server: No such file or directory


In [12]:
import json

config = {
  "provider": {
    "ds4": {
      "name": "ds4.c (local)",
      "npm": "@ai-sdk/openai-compatible",
      "options": {
        "baseURL": "http://127.0.0.1:8000/v1",
        "apiKey": "dsv4-local"
      },
      "models": {
        "deepseek-v4-flash": {
          "name": "DeepSeek V4 Flash (ds4.c local)",
          "limit": {
            "context": 100000,
            "output": 384000
          }
        }
      }
    }
  },
  "agent": {
    "ds4": {
      "description": "DeepSeek V4 Flash served by local ds4-server",
      "model": "ds4/deepseek-v4-flash",
      "temperature": 0
    }
  }
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Configuration saved to config.json')

Configuration saved to config.json


### Step 5: Download Model Weights and Start Server
We need to download the DeepSeek-V4 weights in GGUF format and start the server on port 8000.

In [13]:
import os
# Ensure we are in the correct directory and models folder exists
if not os.path.exists('llama_repo'):
    !git clone https://github.com/ggerganov/llama.cpp.git llama_repo

%cd /content/llama_repo
!mkdir -p models

# Building llama.cpp if not already built
!make -j

Cloning into 'llama_repo'...
remote: Enumerating objects: 92940, done.
remote: Counting objects: 100% (348/348), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 92940 (delta 270), reused 199 (delta 199), pack-reused 92592 (from 3)
Receiving objects: 100% (92940/92940), 386.99 MiB | 19.33 MiB/s, done.
Resolving deltas: 100% (65954/65954), done.
Updating files: 100% (2734/2734), done.
[Errno 2] No such file or directory: '/content/llama_repo'
/content/llama.cpp/llama_repo
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [14]:
# Using huggingface-cli to download a quantized DeepSeek-V4 or similar small model for testing
# Given 12GB RAM, we target a small flash/distill version in GGUF
!pip install huggingface_hub
!huggingface-cli download bartowski/DeepSeek-V3-GGUF --include "*Q2_K.gguf*" --local-dir models --local-dir-use-symlinks False



  A new version of huggingface_hub is available: 1.11.0 → 1.14.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/cli/_cli_utils.py", line 1045, in _prompt_autoupdate
    raw_answer = sys.stdin.readline()
                 ^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/bin/huggingface-cli", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/cli/deprecated_cli.py", line 15, in main
    check_cli_update("huggingface_hub")
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/cli/_cli_utils.py", line 968, in check_cli_update
    _check_cli_update(library)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/cli/_cli_utils.py", line 1006, in _c

In [ ]:
# Start the server on port 8000 with disk-based KV cache
# We run this in the background using 'nohup' so the cell finishes but the server stays up
!nohup ./llama-server --model models/*.gguf --port 8000 --ctx-size 100000 --host 127.0.0.1 > server.log 2>&1 &

In [ ]:
# Verify the server is responding on the configured port
!curl http://127.0.0.1:8000/v1/models

### Step 6: Configure Claude Code Environment
This step sets up the environment variables to route Claude Code requests to our local DeepSeek server.

In [ ]:
env_script = """
export ANTHROPIC_BASE_URL="http://127.0.0.1:8000"
export ANTHROPIC_AUTH_TOKEN="dsv4-local"
export ANTHROPIC_MODEL="deepseek-v4-flash"
export CLAUDE_CODE_SUBAGENT_MODEL="deepseek-v4-flash"
export CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC=1
"""
with open('setup_claude_env.sh', 'w') as f:
    f.write(env_script)

print('Environment setup script created: setup_claude_env.sh')

### Step 7: Final Connectivity Test
Let's perform a dummy completion request to ensure the local provider is fully operational.

In [ ]:
!curl http://127.0.0.1:8000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -H 'Authorization: Bearer dsv4-local' \
  -d '{
    "model": "deepseek-v4-flash",
    "messages": [{"role": "user", "content": "Hello, are you online?"}],
    "max_tokens": 10
  }'